# Pure Solver Distillation (no external data) — NVIDIA Nemotron Reasoning Challenge (final leaderboard 0.74)

A fully **self-contained, winner-independent** solution: trained from the competition's own `train.csv` with **no teacher models and no external datasets**.

Per-category **deterministic solvers** recover each hidden rule directly from the in-context demonstrations and emit a mechanical, step-by-step chain-of-thought. Every trace is verified against ground truth and round-tripped through `\boxed{}`, so the training corpus has **zero label noise**. This is the honest, reproducible-from-scratch baseline — see the repository README for how it compares to the teacher-distillation notebooks.

## 0 · Load the data

Resolve and read the competition `train.csv` — the only data this approach uses.

In [1]:
# Load the competition train.csv (the only data source for this fully self-contained approach).

import os
import polars as pl

_CANDIDATES = [
    "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv",
    "train.csv",
]
TRAIN_CSV = next((p for p in _CANDIDATES if os.path.exists(p)), _CANDIDATES[0])

train = pl.read_csv(TRAIN_CSV)
print(f"Loaded {train.height} rows from {TRAIN_CSV}; columns: {train.columns}")
train.head()

Loaded 9500 rows from train.csv; columns: ['id', 'prompt', 'answer']


id,prompt,answer
str,str,str
"""00066667""","""In Alice's Wonderland, a secre…","""10010111"""
"""000b53cf""","""In Alice's Wonderland, a secre…","""01000011"""
"""00189f6a""","""In Alice's Wonderland, secret …","""cat imagines book"""
"""001b24c4""","""In Alice's Wonderland, numbers…","""XXXVIII"""
"""001c63cb""","""In Alice's Wonderland, secret …","""wizard creates secret"""


## 1 · Load the base model and attach a LoRA adapter

Load the 30B base in bf16 and attach a rank-32 LoRA on the attention + MLP projections. The base weights stay frozen.

In [ ]:
# Load Nemotron-3-Nano-30B (bf16) and attach a rank-32 LoRA on the Mamba + MLP projection
# layers (in_proj / out_proj / up_proj / down_proj). The base model is frozen; only the adapter trains.

import kagglehub
import mamba_ssm
import torch
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer

assert mamba_ssm is not None

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
OUTPUT_DIR = "/kaggle/working"
LORA_RANK = 32
MAX_LENGTH = 8192

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map={"": 0},
    trust_remote_code=True,
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)
model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
print("Base model loaded (bf16, no quantization).")

print(f"Attaching LoRA adapter (rank={LORA_RANK})...")
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=32,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 2 · The data engine — deterministic solvers + verified traces

The heart of this approach: per-category solvers that recover each hidden rule directly from the in-context demonstrations and emit a mechanical chain-of-thought. A trace is kept only if it reproduces the ground-truth answer.

In [2]:
# Per-category deterministic solvers. Each recovers the hidden rule from the in-context demos
# and writes a step-by-step trace; only traces whose \boxed{} answer matches ground truth are
# kept, so the corpus has zero label noise.

import math
import re
import itertools
import numpy as np
from collections import Counter

# --- mechanical long-arithmetic helpers (ported from the #1 solution) -----------
# Every numeric trace below is written so the model only ever reproduces SINGLE-DIGIT
# operations (repeated subtraction / place-value multiplication). It never has to
# "just know" a float quotient or product — that is the fix for the numeric tasks.
def _fmt_int_with_dp(value, dp):
    if dp == 0:
        return str(value)
    s = str(value).zfill(dp + 1)
    s = s[: len(s) - dp] + "." + s[len(s) - dp:]
    s = s.lstrip("0") or "0"
    if s.startswith("."):
        s = "0" + s
    return s


def _truncate_3dp(s):
    if "." not in s:
        return s
    i, f = s.split(".")
    return s if len(f) <= 3 else i + "." + f[:3]


def _dp_count(s):
    return len(s.split(".")[1]) if "." in s else 0


def _pad_dp(s, n):
    if "." not in s:
        s = s + "."
    i, f = s.split(".")
    return i + "." + f.ljust(n, "0")


def _cast_dp_pair(a, b):
    t = max(_dp_count(a), _dp_count(b))
    return _pad_dp(a, t), _pad_dp(b, t)


def long_multiplication_lines(a_str, b_str):
    """Step-by-step place-value multiplication; returns (lines, exact_product_str)."""
    a_dp = len(a_str.split(".")[1]) if "." in a_str else 0
    b_dp = len(b_str.split(".")[1]) if "." in b_str else 0
    total_dp = a_dp + b_dp
    a_int = int(a_str.replace(".", ""))
    b_int = int(b_str.replace(".", ""))
    lines, comps = [], []
    b_digits = str(abs(b_int))
    nd = len(b_digits)
    for i in range(nd - 1, -1, -1):
        d = int(b_digits[i])
        if d == 0:
            continue
        comp_scaled = d * (10 ** (nd - 1 - i))
        comp_display = _fmt_int_with_dp(comp_scaled, b_dp)
        if b_dp > 0:
            comp_display = _pad_dp(comp_display, b_dp)
        product_int = a_int * comp_scaled
        product_display = _fmt_int_with_dp(product_int, total_dp)
        if total_dp > 0:
            product_display = _pad_dp(product_display, total_dp)
        comps.append((comp_display, product_int, product_display))
    for comp_display, _, product_display in comps:
        lines.append(f"{a_str} * {comp_display} = {product_display}")
    if len(comps) >= 2:
        running = comps[0][1]
        for i in range(1, len(comps)):
            rd = _fmt_int_with_dp(running, total_dp)
            if total_dp > 0:
                rd = _pad_dp(rd, total_dp)
            running += comps[i][1]
            sd = _fmt_int_with_dp(running, total_dp)
            if total_dp > 0:
                sd = _pad_dp(sd, total_dp)
            lines.append(f"{rd} + {comps[i][2]} = {sd}")
    return lines, _fmt_int_with_dp(a_int * b_int, total_dp)


def long_division_lines(num_str, den_str, max_decimal_digits=3):
    """Long division via repeated subtraction; returns (lines, quotient_str)."""
    max_dp = max(_dp_count(num_str), _dp_count(den_str))
    num = int(round(float(num_str) * 10 ** max_dp))
    den = int(round(float(den_str) * 10 ** max_dp))
    lines, acc, dd = [], 0, 0

    def fmt_acc():
        if dd == 0:
            return str(acc)
        s = str(acc).zfill(dd + 1)
        return s[:-dd] + "." + s[-dd:]

    def fmt_scale():
        return "1" if dd == 0 else "0." + "0" * (dd - 1) + "1"

    def fmt_line(n):
        return f"= {fmt_acc()} + {fmt_scale()} * {n} / {den}"

    lines.append(fmt_line(num))
    while dd <= max_decimal_digits:
        if num >= den:
            num -= den
            acc += 1
            lines.append(fmt_line(num))
        else:
            dd += 1
            if dd > max_decimal_digits:
                break
            num *= 10
            acc *= 10
            lines.append(fmt_line(num))
    if dd > max_decimal_digits:
        dd = max_decimal_digits
    return lines, fmt_acc()


def _median_str(value_str_pairs):
    """Median by float value, returning the original string (lower-middle if even)."""
    paired = sorted(value_str_pairs)
    if len(paired) % 2 == 0 and len(paired) >= 2:
        return paired[len(paired) // 2 - 1][1]
    return paired[len(paired) // 2][1]



def extract_answer(text: str) -> str:
    """Return the last non-empty \\boxed{...} content."""
    matches = re.findall(r"\\boxed\{([^}]*)(?:\}|$)", text)
    nz = [m.strip() for m in matches if m.strip()]
    if nz:
        return nz[-1]
    return matches[-1].strip() if matches else ""


def compare_answer(stored: str, predicted: str) -> bool:
    """Official scorer: binary->exact, numeric->rel tol 1e-2, else case-insensitive."""
    stored, predicted = str(stored).strip(), str(predicted).strip()
    if re.fullmatch(r"[01]+", stored):
        return predicted.lower() == stored.lower()
    try:
        return math.isclose(float(stored), float(predicted), rel_tol=1e-2, abs_tol=1e-5)
    except Exception:
        return predicted.lower() == stored.lower()


def categorize(prompt: str) -> str:
    """Detect the puzzle category from a keyword in the prompt."""
    if "bit manipulation" in prompt:   return "bit_manipulation"
    if "gravitational" in prompt:      return "gravity"
    if "unit conversion" in prompt:    return "unit_conversion"
    if "encryption" in prompt:         return "cipher"
    if "numeral system" in prompt:     return "numeral"
    if "equations" in prompt:          return "equation"
    return "unknown"


_ROMAN = [(1000, "M"), (900, "CM"), (500, "D"), (400, "CD"), (100, "C"),
          (90, "XC"), (50, "L"), (40, "XL"), (10, "X"), (9, "IX"),
          (5, "V"), (4, "IV"), (1, "I")]


def _to_roman(n: int) -> str:
    s = ""
    for v, sym in _ROMAN:
        while n >= v:
            s += sym; n -= v
    return s


def solve_numeral(prompt):
    """Recover the decimal->Roman rule and convert the query greedily, one symbol at a time."""
    demos = re.findall(r"(\d+)\s*->\s*([IVXLCDM]+)", prompt)
    q = re.search(r"write the number (\d+)", prompt)
    if not q:
        return None
    if demos and not all(_to_roman(int(d)) == r for d, r in demos):
        return None
    n = int(q.group(1)); ans = _to_roman(n)
    lines = [f"This is an Arabic-to-Roman numeral conversion. Converting {n} greedily:"]
    rem, parts = n, []
    for v, sym in _ROMAN:
        while rem >= v:
            lines.append(f"  {rem} >= {v} -> {sym}, remainder {rem - v}")
            parts.append(sym); rem -= v
    lines.append(f"Concatenating {' '.join(parts)} gives {ans}.")
    reasoning = "\n".join(lines)
    return reasoning, ans


def solve_gravity(prompt):
    """Fit d = 0.5*g*t^2 = k*t^2; derive k by long division and the answer by long mult."""
    obs = re.findall(r"t = ([\d.]+)s, distance = ([\d.]+)", prompt)
    q = re.search(r"t = ([\d.]+)s", prompt.split("Now")[-1])
    if not obs or not q:
        return None
    obs = obs[:3]  # 3 observations -> robust median, keeps the trace short enough to batch
    lines = ["We use d = 0.5*g*t^2 = k*t^2. Find k = d / t^2 from each observation."]
    kstrs = []
    for t_str, d_str in obs:
        _, t2 = long_multiplication_lines(t_str, t_str)
        dc, t2c = _cast_dp_pair(_truncate_3dp(d_str), _truncate_3dp(t2))
        lines.append(f"t={t_str}, d={d_str}:  t^2 = {t_str} * {t_str} = {t2}")
        lines.append(f"k = {dc} / {t2c}")
        dl, ks = long_division_lines(dc, t2c)
        lines += dl + [f"= {ks}"]
        kstrs.append(ks)
    med = _median_str([(float(s), s) for s in kstrs])
    lines.append(f"k values: {', '.join(kstrs)}; median k = {med}")
    qs = q.group(1)
    _, tq2 = long_multiplication_lines(qs, qs)
    lines.append(f"For t={qs}: t^2 = {qs} * {qs} = {tq2}")
    disp = med.rstrip("0").rstrip(".") or "0"
    lines.append(f"d = {disp} * {tq2}")
    ml, mres = long_multiplication_lines(disp, tq2)
    lines += ml
    ans = f"{float(mres):.2f}"
    lines.append(f"= {mres}, rounded to 2 decimals = {ans}")
    reasoning = "\n".join(lines)
    return reasoning, ans


def solve_unit(prompt):
    """Fit output = factor*input; derive the factor by long division, convert by long mult."""
    obs = re.findall(r"([\d.]+)\s*m becomes ([\d.]+)", prompt)
    q = re.search(r"convert the following measurement: ([\d.]+)", prompt)
    if not obs or not q:
        return None
    obs = obs[:3]  # 3 observations -> robust median, keeps the trace short enough to batch
    lines = ["Each output is the input times a fixed factor; find it from the examples."]
    fstrs = []
    for i_str, o_str in obs:
        ic, oc = _cast_dp_pair(_truncate_3dp(i_str), _truncate_3dp(o_str))
        lines.append(f"{i_str} -> {o_str}:  factor = {oc} / {ic}")
        dl, fs = long_division_lines(oc, ic)
        lines += dl + [f"= {fs}"]
        fstrs.append(fs)
    med = _median_str([(float(s), s) for s in fstrs])
    lines.append(f"factor values: {', '.join(fstrs)}; median factor = {med}")
    qs = q.group(1)
    disp = med.rstrip("0").rstrip(".") or "0"
    lines.append(f"Converting {qs}: {qs} * {disp}")
    ml, mres = long_multiplication_lines(qs, disp)
    lines += ml
    ans = f"{float(mres):.2f}"
    lines.append(f"= {mres}, rounded to 2 decimals = {ans}")
    reasoning = "\n".join(lines)
    return reasoning, ans


WONDERLAND_VOCAB = [
    "above", "alice", "ancient", "around", "beyond", "bird", "book", "bright",
    "castle", "cat", "cave", "chases", "clever", "colorful", "creates", "crystal",
    "curious", "dark", "discovers", "door", "dragon", "draws", "dreams", "explores",
    "follows", "forest", "found", "garden", "golden", "hatter", "hidden", "imagines",
    "in", "inside", "island", "key", "king", "knight", "library", "magical", "map",
    "message", "mirror", "mountain", "mouse", "mysterious", "near", "ocean", "palace",
    "potion", "princess", "puzzle", "queen", "rabbit", "reads", "school", "secret",
    "sees", "silver", "story", "strange", "student", "studies", "teacher", "the",
    "through", "tower", "treasure", "turtle", "under", "valley", "village", "watches",
    "wise", "wizard", "wonderland", "writes",
]


def _cipher_match_word(cw, mapping, vocab_by_len):
    """Vocabulary words whose letters are consistent with the partial cipher map."""
    cands = set()
    for v in vocab_by_len.get(len(cw), ()):
        ok = True; used = {}
        for c, pl_ in zip(cw, v):
            if c in mapping:
                if mapping[c] != pl_:
                    ok = False; break
            else:
                if c in used and used[c] != pl_:
                    ok = False; break
                used[c] = pl_
        if ok:
            cands.add(v)
    return cands


def solve_cipher(prompt, vocab=WONDERLAND_VOCAB):
    """Build the substitution from demos; resolve unseen letters via the vocabulary."""
    lines = re.findall(r"^(.*?) -> (.*?)$", prompt, re.M)
    qm = re.search(r"decrypt the following text: (.+)", prompt)
    if not qm:
        return None
    q = qm.group(1).strip()
    mapping = {}
    for ct, pt in lines:
        ct, pt = ct.strip(), pt.strip()
        if len(ct) != len(pt):
            return None
        for a, b in zip(ct, pt):
            if a.isalpha() and b.isalpha():
                if a in mapping and mapping[a] != b:
                    return None
                mapping[a] = b
    vocab_by_len = {}
    for w in vocab:
        vocab_by_len.setdefault(len(w), []).append(w)
    out_words = []
    word_steps = []
    inferred = False
    for cw in q.split():
        if not cw.isalpha():
            out_words.append(cw)
            word_steps.append((cw, cw, False))
            continue
        if all(c in mapping for c in cw):
            pw = "".join(mapping[c] for c in cw)
            out_words.append(pw)
            word_steps.append((cw, pw, False))
        else:
            cands = _cipher_match_word(cw, mapping, vocab_by_len)
            if len(cands) == 1:
                w = next(iter(cands))
                for c, pl_ in zip(cw, w):
                    mapping.setdefault(c, pl_)
                out_words.append(w)
                word_steps.append((cw, w, True))
                inferred = True
            else:
                return None
    ans = " ".join(out_words)
    qletters = sorted({c for cw, _, _ in word_steps for c in cw if c in mapping})
    map_str = ", ".join(f"{c}->{mapping[c]}" for c in qletters)
    step_lines = []
    for cw, pw, was_inf in word_steps:
        if not cw.isalpha():
            continue
        pairs = " ".join(f"{c}->{p}" for c, p in zip(cw, pw))
        tag = "  (recognised from the Wonderland vocabulary)" if was_inf else ""
        step_lines.append(f"  {'-'.join(cw)}  =>  {pairs}  =>  {pw}{tag}")
    note = (
        "\nSome query letters never appear in the examples; the word they spell is "
        "recognised from the Wonderland vocabulary, which pins down those letters."
        if inferred else "")
    reasoning = (
        "The examples define a letter-substitution cipher; aligning each ciphertext "
        "letter with its plaintext letter gives a consistent map: " + map_str + "." + note + "\n"
        "Decrypting the query word by word, one letter at a time:\n"
        + "\n".join(step_lines) + "\n"
        f"So '{q}' decrypts to '{ans}'."
    )
    return reasoning, ans


def _bit_rule(col, X, n):
    """Find the simplest Boolean function of input bits matching an output-bit column."""
    m = len(X)
    if all(c == 0 for c in col): return ("0", lambda qb: 0)
    if all(c == 1 for c in col): return ("1", lambda qb: 1)
    for j in range(n):
        if all(col[k] == X[k][j] for k in range(m)):
            return (f"in[{j}]", lambda qb, j=j: qb[j])
        if all(col[k] == 1 - X[k][j] for k in range(m)):
            return (f"NOT in[{j}]", lambda qb, j=j: 1 - qb[j])
    ops = [("XOR", lambda x, y: x ^ y), ("XNOR", lambda x, y: 1 - (x ^ y)),
           ("AND", lambda x, y: x & y), ("OR", lambda x, y: x | y),
           ("NAND", lambda x, y: 1 - (x & y)), ("NOR", lambda x, y: 1 - (x | y)),
           ("AND-NOT", lambda x, y: x & (1 - y)), ("OR-NOT", lambda x, y: x | (1 - y))]
    for a, b in itertools.permutations(range(n), 2):
        for nm, fn in ops:
            if all(col[k] == fn(X[k][a], X[k][b]) for k in range(m)):
                return (f"in[{a}] {nm} in[{b}]", lambda qb, fn=fn, a=a, b=b: fn(qb[a], qb[b]))
    for a, b, c in itertools.combinations(range(n), 3):
        if all(col[k] == (1 if X[k][a] + X[k][b] + X[k][c] >= 2 else 0) for k in range(m)):
            return (f"majority(in[{a}],in[{b}],in[{c}])",
                    lambda qb, a=a, b=b, c=c: 1 if qb[a] + qb[b] + qb[c] >= 2 else 0)
    for a in range(n):
        for b in range(n):
            for c in range(n):
                if all(col[k] == (X[k][b] if X[k][a] else X[k][c]) for k in range(m)):
                    return (f"in[{a}]?in[{b}]:in[{c}]",
                            lambda qb, a=a, b=b, c=c: qb[b] if qb[a] else qb[c])
    A = np.array([list(r) + [1] for r in X]) % 2
    y = np.array(col) % 2
    Aug = np.concatenate([A, y.reshape(-1, 1)], 1) % 2
    R, cn = Aug.shape; r = 0; piv = {}
    for cc in range(cn - 1):
        pr = next((rr for rr in range(r, R) if Aug[rr, cc]), None)
        if pr is None: continue
        Aug[[r, pr]] = Aug[[pr, r]]
        for rr in range(R):
            if rr != r and Aug[rr, cc]: Aug[rr] = (Aug[rr] + Aug[r]) % 2
        piv[cc] = r; r += 1
    for rr in range(R):
        if Aug[rr, :-1].sum() == 0 and Aug[rr, -1] == 1:
            return None
    w = np.zeros(cn - 1, dtype=int)
    for cc, rr in piv.items():
        w[cc] = Aug[rr, -1]
    terms = [f"in[{j}]" for j in range(n) if w[j]]
    label = ("XOR(" + ",".join(terms) + ")" if terms else "0") + ("^1" if w[n] else "")
    return (label, lambda qb, w=w, n=n: int((np.array(list(qb) + [1]) % 2).dot(w) % 2))


_BIT_OPS = [("XOR", lambda x, y: x ^ y), ("XNOR", lambda x, y: 1 - (x ^ y)),
            ("AND", lambda x, y: x & y), ("OR", lambda x, y: x | y),
            ("NAND", lambda x, y: 1 - (x & y)), ("NOR", lambda x, y: 1 - (x | y)),
            ("AND-NOT", lambda x, y: x & (1 - y)), ("OR-NOT", lambda x, y: x | (1 - y))]


def _bit_candidates(col, X, i, n=8):
    """ALL grammar rules consistent with an output-bit column (simplest first), each
    tagged with a relative-offset feature so recurring cross-bit patterns can be
    preferred: the puzzle generators apply ONE shifted pattern to every output bit,
    so the demo-consistent candidate whose (op, i-relative operands) recurs across
    bits is far more likely to be the true rule than the lexicographically first."""
    m = len(X)
    out = []
    if all(c == 0 for c in col):
        out.append(("0", ("const", 0), lambda qb: 0))
    if all(c == 1 for c in col):
        out.append(("1", ("const", 1), lambda qb: 1))
    for j in range(n):
        if all(col[k] == X[k][j] for k in range(m)):
            out.append((f"in[{j}]", ("copy", (j - i) % n), lambda qb, j=j: qb[j]))
        if all(col[k] == 1 - X[k][j] for k in range(m)):
            out.append((f"NOT in[{j}]", ("not", (j - i) % n), lambda qb, j=j: 1 - qb[j]))
    for a, b in itertools.permutations(range(n), 2):
        for nm, fn in _BIT_OPS:
            if all(col[k] == fn(X[k][a], X[k][b]) for k in range(m)):
                out.append((f"in[{a}] {nm} in[{b}]", (nm, (a - i) % n, (b - i) % n),
                            lambda qb, fn=fn, a=a, b=b: fn(qb[a], qb[b])))
    for a, b, c in itertools.combinations(range(n), 3):
        if all(col[k] == (1 if X[k][a] + X[k][b] + X[k][c] >= 2 else 0) for k in range(m)):
            out.append((f"majority(in[{a}],in[{b}],in[{c}])",
                        ("maj", (a - i) % n, (b - i) % n, (c - i) % n),
                        lambda qb, a=a, b=b, c=c: 1 if qb[a] + qb[b] + qb[c] >= 2 else 0))
    return out


def _feat_pattern(feat):
    """Render a relative-offset feature as a per-bit pattern string."""
    kind = feat[0]
    if kind == "const":
        return f"out[i] = {feat[1]}"
    if kind == "copy":
        return f"out[i] = in[(i+{feat[1]})%8]"
    if kind == "not":
        return f"out[i] = NOT in[(i+{feat[1]})%8]"
    if kind == "maj":
        return ("out[i] = majority(" +
                ", ".join(f"in[(i+{o})%8]" for o in feat[1:]) + ")")
    return f"out[i] = in[(i+{feat[1]})%8] {kind} in[(i+{feat[2]})%8]"


def solve_bit(prompt):
    """Fit each output bit to a Boolean function of the input bits. Among all
    demo-consistent candidates per bit, prefer the relative pattern that RECURS
    across output bits (coherence — measured +27 pts of verified coverage vs
    simplest-first); per-bit GF(2)/choice fallback when the grammar finds nothing.
    The trace shows the column evidence, then applies the rules to the query."""
    pairs = re.findall(r"([01]{8})\s*->\s*([01]{8})", prompt)
    qm = re.search(r"output for:\s*([01]{8})", prompt)
    if not pairs or not qm:
        return None
    q = qm.group(1)
    X = [[int(c) for c in i] for i, o in pairs]
    Y = [[int(c) for c in o] for i, o in pairs]
    qb = [int(c) for c in q]
    cands = [_bit_candidates([Y[k][ob] for k in range(len(pairs))], X, ob)
             for ob in range(8)]
    feat_count = Counter()
    for c in cands:
        for f in {feat for _, feat, _ in c}:
            feat_count[f] += 1
    rules = []
    for ob in range(8):
        if cands[ob]:
            mc = max(feat_count[f] for _, f, _ in cands[ob])
            if mc >= 2:   # a pattern shared by 2+ bits beats "simplest first"
                lbl, _, fn = next(t for t in cands[ob] if feat_count[t[1]] == mc)
            else:
                lbl, _, fn = cands[ob][0]
            rules.append((lbl, fn))
        else:
            col = [Y[k][ob] for k in range(len(pairs))]
            r = _bit_rule(col, X, 8)
            if r is None:
                return None
            rules.append(r)
    out = [fn(qb) for _, fn in rules]
    ans = "".join(str(b) for b in out)
    invals = ", ".join(f"in[{j}]={qb[j]}" for j in range(8))

    # Derivation: column evidence for the non-trivial multi-input rules (the model
    # learns to FIND the rule, not just apply a stated one); trivial rules tersely.
    deriv = []
    for ob, (lbl, _) in enumerate(rules):
        positions = sorted({int(p) for p in re.findall(r"in\[(\d)\]", lbl)})
        if len(positions) >= 2:
            nshow = min(len(pairs), 4)
            hdr = ",".join(f"in[{p}]" for p in positions)
            ev = ", ".join(
                "(" + ",".join(str(X[k][p]) for p in positions) + ")->" + str(Y[k][ob])
                for k in range(nshow)
            )
            deriv.append(f"  out[{ob}] = {lbl}: ({hdr})->out[{ob}] for the examples = {ev}, ... -> consistent")
        else:
            deriv.append(f"  out[{ob}] = {lbl}")

    top_feat, top_n = feat_count.most_common(1)[0] if feat_count else (None, 0)
    pattern_note = (
        f"Across the output bits the same relative pattern recurs "
        f"({_feat_pattern(top_feat)}); prefer it when several rules fit a column.\n"
        if top_n >= 3 else "")

    desc = "; ".join(f"out[{i}]={lbl}" for i, (lbl, _) in enumerate(rules))
    steps = []
    for i, (lbl, _) in enumerate(rules):
        if "in[" in lbl:
            subst = re.sub(r"in\[(\d)\]", lambda m: str(qb[int(m.group(1))]), lbl)
            steps.append(f"  out[{i}] = {lbl} = {subst} = {out[i]}")
        else:
            steps.append(f"  out[{i}] = {lbl} = {out[i]}")

    reasoning = (
        "Read the examples as bit columns. For each output position, find the Boolean "
        "function of the input bits that matches every example:\n"
        + "\n".join(deriv) + "\n"
        + pattern_note
        + "So the per-bit rules are: " + desc + ".\n"
        f"Now substitute the query input {q} ({invals}) and evaluate each output bit:\n"
        + "\n".join(steps) + "\n"
        f"Reading out[0..7] in order gives {ans}."
    )
    return reasoning, ans


def _rev_num(s):
    """Reverse a numeric string, keeping a leading minus sign in place."""
    return ("-" + s[1:][::-1]) if s.startswith("-") else s[::-1]


def _num_base(sa, sb):
    """Candidate arithmetic/digit operations on two 2-digit operand strings."""
    a, b = int(sa), int(sb)
    o = [("concatenation", sa + sb), ("reverse concatenation", sb + sa),
         ("addition", str(a + b)), ("absolute difference", str(abs(a - b))),
         ("negated absolute difference", str(-abs(a - b))),
         ("subtraction", str(a - b)), ("reverse subtraction", str(b - a)),
         ("multiplication", str(a * b)), ("multiply+1", str(a * b + 1)),
         ("multiply-1", str(a * b - 1)), ("add+1", str(a + b + 1)),
         ("add-1", str(a + b - 1)), ("sub+1", str(a - b + 1)), ("sub-1", str(a - b - 1))]
    if a and b:
        o.append(("max mod min", str(max(a, b) % min(a, b))))
    if b:
        o += [("integer division", str(a // b)), ("modulo", str(a % b))]
    if a:
        o += [("reverse division", str(b // a)), ("reverse modulo", str(b % a))]
    if len(sa) == 2 and len(sb) == 2:
        d1, d2, d3, d4 = int(sa[0]), int(sa[1]), int(sb[0]), int(sb[1])
        o += [("digit absolute diff", str(abs(d1 - d3)) + str(abs(d2 - d4))),
              ("digit add mod10", str((d1 + d3) % 10) + str((d2 + d4) % 10)),
              ("digit sub mod10", str((d1 - d3) % 10) + str((d2 - d4) % 10)),
              ("cross multiply", str(d1 * d3 + d2 * d4)),
              ("cross multiply rev", str(d1 * d4 + d2 * d3)),
              ("digit multiply", str(d1 * d3) + str(d2 * d4)),
              ("digit multiply rev", str(d1 * d4) + str(d2 * d3)),
              ("digit sum diff", str((d1 + d2) - (d3 + d4))),
              ("digit sum sum", str((d1 + d2) + (d3 + d4))),
              ("digit product diff", str(d1 * d2 - d3 * d4)),
              ("digit product sum", str(d1 * d2 + d3 * d4)),
              ("determinant", str(d1 * d4 - d2 * d3)),
              ("abs determinant", str(abs(d1 * d4 - d2 * d3)))]
    return o


def _num_cands(sa, sb):
    """All numeric candidates, also with reversed operands and/or reversed result."""
    res = {}
    for rop in (0, 1):
        ta, tb = (sa[::-1], sb[::-1]) if rop else (sa, sb)
        for name, val in _num_base(ta, tb):
            for rr in (0, 1):
                v = _rev_num(val) if rr else val
                tag = name + (" [rev operands]" if rop else "") + (" [rev result]" if rr else "")
                res[tag] = v
    return res


def _parse_equation(prompt):
    """Parse 'AB op CD = out' demo lines and the query string."""
    demos = []
    for ln in prompt.split("\n"):
        s = ln.strip().strip("`").strip()
        if " = " in s and "determine" not in s:
            left, right = s.split(" = ", 1)
            demos.append((left.strip(), right.strip()))
    qm = re.search(r"determine the result for:\s*(.+)", prompt)
    q = qm.group(1).strip().strip("`").strip() if qm else None
    return demos, q


def _eq_explain(opname, a, b, ans):
    """Reproducible arithmetic lines for the common ops; else just state the result."""
    if opname == "addition":
        return [f"{a} + {b} = {ans}"]
    if opname == "subtraction (a-b)":
        return [f"{a} - {b} = {ans}"]
    if opname == "reverse subtraction (b-a)":
        return [f"{b} - {a} = {ans}"]
    if opname == "absolute difference":
        return [f"|{a} - {b}| = {ans}"]
    if opname == "concatenation":
        return [f"writing {a} then {b} gives {ans}"]
    if opname == "reverse concatenation":
        return [f"writing {b} then {a} gives {ans}"]
    if opname == "multiplication":
        lines = [f"{a} * {b}:"]
        ml, prod = long_multiplication_lines(a, b)
        lines += ml + [f"= {prod}"]
        return lines
    return [f"Applying it to {a} and {b} gives {ans}."]



# Empirical prior over TRUE equation ops, measured on the train rows the solver
# verifies (a learnable global prior, like WONDERLAND_VOCAB — the trace still checks
# the op against the demos). Re-ranking candidates by it fixes rows where several
# ops fit the demos and dict order picked the wrong one (+35 verified rows).
_EQ_OP_PRIOR = {op: r for r, op in enumerate([
    "multiplication [rev operands] [rev result]", "addition [rev operands] [rev result]",
    "concatenation", "reverse concatenation", "absolute difference",
    "multiply-1 [rev operands] [rev result]", "multiply+1 [rev operands] [rev result]",
    "add-1 [rev operands] [rev result]", "multiplication", "addition",
    "add+1 [rev operands] [rev result]", "multiply+1", "add+1", "add-1",
    "absolute difference [rev operands] [rev result]", "multiply-1",
    "negated absolute difference [rev operands] [rev result]",
    "max mod min [rev operands] [rev result]", "max mod min",
    "subtraction [rev operands] [rev result]", "negated absolute difference",
    "digit absolute diff", "subtraction",
])}

def solve_equation(prompt):
    """Deduce the per-puzzle operation for the query operator and apply it, showing the math."""
    demos, q = _parse_equation(prompt)
    if not q or len(q) != 5:
        return None
    qop = q[2]
    same = [(i, o) for i, o in demos if len(i) == 5 and i[2] == qop]
    if not same:
        return None
    isdig = all(q[k].isdigit() for k in (0, 1, 3, 4))
    if isdig:
        same = [(i, o) for i, o in same if all(i[k].isdigit() for k in (0, 1, 3, 4))]
        if not same:
            return None
        cand_keys = sorted(_num_cands(same[0][0][0:2], same[0][0][3:5]).keys(),
                           key=lambda k: _EQ_OP_PRIOR.get(k, len(_EQ_OP_PRIOR)))
        for k in cand_keys:
            if all(_num_cands(i[0:2], i[3:5]).get(k) == o for i, o in same):
                ans = _num_cands(q[0:2], q[3:5]).get(k)
                if ans is None:
                    return None
                lines = [
                    f"Each puzzle assigns one operation to the operator '{qop}'. "
                    f"Testing the examples that use '{qop}', the consistent operation is '{k}'.",
                ]
                # Confirm the rule on the demos that share this operator.
                for i, o in same:
                    lines.append(f"check: {i[0:2]}{qop}{i[3:5]} -> {o}")
                lines.append(f"Applying it to {q[0:2]}{qop}{q[3:5]}:")
                lines += _eq_explain(k, q[0:2], q[3:5], ans)
                lines.append(f"so the result is {ans}.")
                return "\n".join(lines), ans
        return None
    for kind in ("concatenation", "reverse concatenation"):
        def app(s):
            return s[0] + s[1] + s[3] + s[4] if kind == "concatenation" else s[3] + s[4] + s[0] + s[1]
        if all(app(i) == o for i, o in same):
            val = app(q)
            joined = f"{q[3:5]} then {q[0:2]}" if kind == "reverse concatenation" else f"{q[0:2]} then {q[3:5]}"
            reasoning = (
                f"The operator '{qop}' joins the two operand pairs; the examples show "
                f"this is {kind}.\n"
                f"Writing {joined} gives {val}."
            )
            return reasoning, val
    return None


SOLVERS = {
    "numeral": solve_numeral,
    "gravity": solve_gravity,
    "unit_conversion": solve_unit,
    "cipher": solve_cipher,
    "bit_manipulation": solve_bit,
    "equation": solve_equation,
}


def build_corpus(df):
    """Run solvers over the dataframe; keep only verified, round-trip-parseable traces."""
    records = []
    kept = {c: 0 for c in SOLVERS}
    seen = {c: 0 for c in SOLVERS}
    for row in df.iter_rows(named=True):
        cat = categorize(row["prompt"])
        solver = SOLVERS.get(cat)
        if solver is None:
            continue
        seen[cat] += 1
        out = solver(row["prompt"])
        if out is None:
            continue
        reasoning, ans = out
        completion = f"{reasoning}\n</think>\n\\boxed{{{ans}}}"
        if not compare_answer(ans, extract_answer(completion)):
            continue
        if compare_answer(row["answer"], ans):
            records.append(dict(id=row["id"], category=cat, prompt=row["prompt"],
                                reasoning=reasoning, answer=str(row["answer"]).strip()))
            kept[cat] += 1
    print(f"{'category':18s} {'kept':>6} {'seen':>6} {'cov%':>6}")
    for c in SOLVERS:
        cov = 100 * kept[c] / seen[c] if seen[c] else 0
        print(f"{c:18s} {kept[c]:>6} {seen[c]:>6} {cov:>5.1f}")
    print(f"TOTAL verified traces: {len(records)} / {df.height} "
          f"({100 * len(records) / df.height:.1f}%)")
    return records


corpus_records = build_corpus(train)

category             kept   seen   cov%
numeral              1576   1576 100.0
gravity              1597   1597 100.0
unit_conversion      1594   1594 100.0
cipher               1566   1576  99.4
bit_manipulation      873   1602  54.5
equation              512   1555  32.9
TOTAL verified traces: 7718 / 9500 (81.2%)


## 3 · Augmentation — synthetic puzzles + atomic sub-skills

Extra verified-synthetic puzzles plus small single-skill examples (spelling, bit columns, etc.) to reinforce the mechanics the solvers rely on. Everything passes the same verify + round-trip filter.

In [3]:
# Verified-synthetic augmentation: extra generated puzzles + atomic single-skill examples,
# all passed through the same verify + round-trip filter as the real traces.

import random

AUGMENT = True
AUG_CIPHER = 300
AUG_BIT = 500   # bit real coverage is 82% now; fewer synthetics needed
AUG_EQUATION = 1100
AUG_SKILLS = True
AUG_SPELL = 800
AUG_CONCAT = 500
AUG_SPLIT = 500
AUG_LSTRIP = 300
AUG_MATCH = 800

_CIPHER_TMPL = ("In Alice's Wonderland, secret encryption rules are used on text. "
                "Here are some examples:\n{demos}\nNow, decrypt the following text: {q}")
_BIT_TMPL = ("In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit "
             "binary numbers. The transformation involves operations like bit shifts, "
             "rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.\n\n"
             "Here are some examples of input -> output:\n{demos}\n\n"
             "Now, determine the output for: {q}")
_EQ_TMPL = ("In Alice's Wonderland, a secret set of transformation rules is applied to "
            "equations. Below are a few examples:\n{demos}\nNow, determine the result for: {q}")
_ABC = "abcdefghijklmnopqrstuvwxyz"
_EQ_SYMS = list("!\"#$%&'()*+-./:;<>?@[\\]^`{|}")


def _rand_key(rng):
    """Random derangement of a-z as a plaintext->ciphertext substitution."""
    letters = list(_ABC)
    while True:
        shuf = letters[:]
        rng.shuffle(shuf)
        if all(a != b for a, b in zip(letters, shuf)):
            return dict(zip(letters, shuf))


def aug_cipher(rng):
    """Synthesize a substitution-cipher puzzle whose query needs vocabulary inference."""
    enc = _rand_key(rng)
    sentence = lambda n: " ".join(rng.choice(WONDERLAND_VOCAB) for _ in range(n))
    demos = [sentence(rng.randint(3, 5)) for _ in range(5)]
    demo_letters = set("".join(demos))
    q = sentence(rng.randint(2, 4))
    for _ in range(40):
        cand = sentence(rng.randint(2, 4))
        if not (set(cand) <= demo_letters):
            q = cand; break
    enc_line = lambda s: "".join(enc.get(c, c) for c in s)
    demo_str = "\n".join(f"{enc_line(s)} -> {s}" for s in demos)
    return _CIPHER_TMPL.format(demos=demo_str, q=enc_line(q)), q


def _rotate(b, k, left):
    k %= 8
    if k == 0:
        return b[:]
    return b[k:] + b[:k] if left else b[-k:] + b[:-k]


def aug_bit(rng):
    """Synthesize an 8-bit puzzle from a random rule in the solver's grammar."""
    fam = rng.choice(["rotate", "shift", "xor", "not", "perbit", "gates", "majority"])
    k = rng.randint(1, 7); left = rng.random() < 0.5
    mask = [rng.randint(0, 1) for _ in range(8)]
    if fam == "perbit":
        spec = [(rng.randrange(8), rng.random() < 0.5) for _ in range(8)]
    elif fam == "gates":
        spec = [(rng.randrange(8), rng.randrange(8),
                 rng.choice(["and", "or", "xor", "andnot", "ornot"])) for _ in range(8)]
    elif fam == "majority":
        spec = [tuple(rng.sample(range(8), 3)) for _ in range(8)]
    else:
        spec = None

    def apply(inb):
        if fam == "rotate":
            return _rotate(inb, k, left)
        if fam == "shift":
            return inb[k:] + [0] * k if left else [0] * k + inb[:8 - k]
        if fam == "xor":
            return [x ^ m for x, m in zip(inb, mask)]
        if fam == "not":
            return [1 - x for x in inb]
        if fam == "perbit":
            return [(1 - inb[j]) if neg else inb[j] for (j, neg) in spec]
        if fam == "gates":
            g = {"and": lambda x, y: x & y, "or": lambda x, y: x | y, "xor": lambda x, y: x ^ y,
                 "andnot": lambda x, y: x & (1 - y), "ornot": lambda x, y: x | (1 - y)}
            return [g[op](inb[a], inb[b]) for (a, b, op) in spec]
        return [1 if inb[a] + inb[b] + inb[c] >= 2 else 0 for (a, b, c) in spec]

    seen = set(); ins = []
    while len(ins) < 9:
        v = tuple(rng.randint(0, 1) for _ in range(8))
        if v not in seen:
            seen.add(v); ins.append(list(v))
    q = ins[0]
    while tuple(q) in seen:
        q = [rng.randint(0, 1) for _ in range(8)]
    fmt = lambda b: "".join(map(str, b))
    demo_str = "\n".join(f"{fmt(i)} -> {fmt(apply(i))}" for i in ins)
    return _BIT_TMPL.format(demos=demo_str, q=fmt(q)), fmt(apply(q))


def aug_equation(rng):
    """Synthesize a digit-equation puzzle with random per-operator operations."""
    ops = rng.sample(_EQ_SYMS, rng.randint(2, 4))
    base = [n for n, _ in _num_base("12", "34")]
    meaning = {op: rng.choice(base) for op in ops}
    val = lambda a, b, name: dict(_num_base(a, b)).get(name)
    qop = rng.choice(ops)
    lines = []; qcount = 0
    for _ in range(rng.randint(4, 6)):
        op = rng.choice(ops)
        a, b = f"{rng.randint(10, 99)}", f"{rng.randint(10, 99)}"
        v = val(a, b, meaning[op])
        if v is None:
            continue
        lines.append(f"{a}{op}{b} = {v}")
        qcount += op == qop
    while qcount < 2:
        a, b = f"{rng.randint(10, 99)}", f"{rng.randint(10, 99)}"
        v = val(a, b, meaning[qop])
        if v is None:
            return None
        lines.append(f"{a}{qop}{b} = {v}"); qcount += 1
    rng.shuffle(lines)
    qa, qb = f"{rng.randint(10, 99)}", f"{rng.randint(10, 99)}"
    true = val(qa, qb, meaning[qop])
    if true is None:
        return None
    return _EQ_TMPL.format(demos="\n".join(lines), q=f"{qa}{qop}{qb}"), true


_EN = "–"
_LB, _RB = "【", "】"
_SAFE_SYMS = list("!\"#$%&'()*+-./:;<=>?@[]^_|~")
_BIT_OPS = [
    ("AND", lambda x, y: x & y), ("OR", lambda x, y: x | y),
    ("XOR", lambda x, y: x ^ y), ("NAND", lambda x, y: 1 - (x & y)),
    ("NOR", lambda x, y: 1 - (x | y)), ("XNOR", lambda x, y: 1 - (x ^ y)),
    ("AND-NOT", lambda x, y: x & (1 - y)), ("OR-NOT", lambda x, y: x | (1 - y)),
]

_SPELL_TMPL = ("In Alice's Wonderland, a secret rule spells text out one character "
               "at a time. Here are some examples:\n{demos}\n"
               "Now, spell out the following text: {q}")
_CONCAT_TMPL = ("In Alice's Wonderland, a secret rule merges individually bracketed "
                "symbols into one bracket. Here are some examples:\n{demos}\n"
                "Now, merge the following: {q}")
_SPLIT_TMPL = ("In Alice's Wonderland, a secret rule splits one bracketed string into "
               "individually bracketed symbols. Here are some examples:\n{demos}\n"
               "Now, split the following: {q}")
_LSTRIP_TMPL = ("In Alice's Wonderland, a secret rule removes leading spaces from a "
                "bracketed string. Here are some examples:\n{demos}\n"
                "Now, process the following: {q}")
_MATCH_TMPL = ("In Alice's Wonderland, each output bit is a fixed Boolean function of "
               "the two aligned input bits. Here are some examples (A, B -> output):\n"
               "{demos}\nNow, determine the output for: {q}")


def _spell_out(text):
    """Break text into characters (dropping spaces), wrapped/joined with en-dashes."""
    chars = [c for c in text if c != " "]
    return _EN + _EN.join(chars) + _EN


def _spell_phrase(rng):
    """A short Wonderland phrase (reinforces the cipher vocab) or a random letter run."""
    if rng.random() < 0.6:
        return " ".join(rng.choice(WONDERLAND_VOCAB) for _ in range(rng.randint(1, 3)))
    return "".join(rng.choice(_ABC) for _ in range(rng.randint(3, 8)))


def aug_spell(rng):
    """Synthesize a character-spelling puzzle: text -> en-dash separated characters."""
    demos = [_spell_phrase(rng) for _ in range(3)]
    q = _spell_phrase(rng)
    demo_str = "\n".join(f"{s} -> {_spell_out(s)}" for s in demos)
    return _SPELL_TMPL.format(demos=demo_str, q=q), _spell_out(q)


def solve_spell(prompt):
    """Verify the spelling rule from demos and spell the query character by character."""
    lines = re.findall(r"^(.*?) -> (.*?)$", prompt, re.M)
    qm = re.search(r"spell out the following text: (.+)", prompt)
    if not qm or not lines:
        return None
    for inp, out in lines:
        if _spell_out(inp.strip()) != out.strip():
            return None
    q = qm.group(1).strip()
    chars = [c for c in q if c != " "]
    ans = _spell_out(q)
    reasoning = (
        f"The rule writes each character separated by '{_EN}', dropping spaces.\n"
        f"Reading '{q}' character by character: " + ", ".join(chars) + ".\n"
        f"Joining them with '{_EN}' gives {ans}."
    )
    return reasoning, ans


def _box_indiv(chars):
    return "".join(f"{_LB}{c}{_RB}" for c in chars)


def _box_merged(chars):
    return f"{_LB}{''.join(chars)}{_RB}"


def _rand_syms(rng):
    return [rng.choice(_SAFE_SYMS) for _ in range(rng.randint(2, 8))]


def aug_concat(rng):
    """Synthesize a concatenation puzzle: bracketed chars -> one merged bracket."""
    demos = [_rand_syms(rng) for _ in range(3)]
    q = _rand_syms(rng)
    demo_str = "\n".join(f"{_box_indiv(c)} -> {_box_merged(c)}" for c in demos)
    return _CONCAT_TMPL.format(demos=demo_str, q=_box_indiv(q)), _box_merged(q)


def aug_split(rng):
    """Synthesize a splitting puzzle: one merged bracket -> individually bracketed chars."""
    demos = [_rand_syms(rng) for _ in range(3)]
    q = _rand_syms(rng)
    demo_str = "\n".join(f"{_box_merged(c)} -> {_box_indiv(c)}" for c in demos)
    return _SPLIT_TMPL.format(demos=demo_str, q=_box_merged(q)), _box_indiv(q)


def _parse_indiv(s):
    return re.findall(rf"{_LB}(.){_RB}", s)


def _parse_merged(s):
    m = re.search(rf"{_LB}(.*?){_RB}", s)
    return list(m.group(1)) if m else None


def solve_concat(prompt):
    """Read each bracketed symbol in order and wrap them in a single bracket."""
    qm = re.search(r"merge the following: (.+)", prompt)
    if not qm:
        return None
    chars = _parse_indiv(qm.group(1).strip())
    if not chars:
        return None
    ans = _box_merged(chars)
    reasoning = (
        "Each bracketed symbol is read in order and the brackets are dropped, then "
        "all symbols are wrapped in a single bracket.\n"
        "Reading the symbols: " + ", ".join(chars) + ".\n"
        f"Merging them gives {ans}."
    )
    return reasoning, ans


def solve_split(prompt):
    """Open the single bracket and wrap each symbol inside in its own bracket."""
    qm = re.search(r"split the following: (.+)", prompt)
    if not qm:
        return None
    chars = _parse_merged(qm.group(1).strip())
    if not chars:
        return None
    ans = _box_indiv(chars)
    reasoning = (
        "The single bracket is opened and each symbol inside is wrapped in its own "
        "bracket, in order.\n"
        "The symbols inside are: " + ", ".join(chars) + ".\n"
        f"Wrapping each gives {ans}."
    )
    return reasoning, ans


def _rand_syms_str(rng):
    return "".join(rng.choice(_SAFE_SYMS) for _ in range(rng.randint(1, 8)))


def aug_lstrip(rng):
    """Synthesize an lstrip puzzle: a bracketed string with leading spaces -> trimmed."""
    demos = [_rand_syms_str(rng) for _ in range(3)]
    q = _rand_syms_str(rng)
    pad = lambda: " " * rng.randint(1, 3)
    demo_str = "\n".join(f"{_LB}{pad()}{s}{_RB} -> {_LB}{s}{_RB}" for s in demos)
    return _LSTRIP_TMPL.format(demos=demo_str, q=f"{_LB}{pad()}{q}{_RB}"), f"{_LB}{q}{_RB}"


def solve_lstrip(prompt):
    """Strip leading spaces from inside the query bracket."""
    qm = re.search(r"process the following: (.+)", prompt)
    if not qm:
        return None
    m = re.search(rf"{_LB}(.*?){_RB}", qm.group(1).strip())
    if not m:
        return None
    stripped = m.group(1).lstrip(" ")
    ans = f"{_LB}{stripped}{_RB}"
    reasoning = (
        "The rule keeps the bracket but removes any spaces at the start of its "
        "contents.\n"
        f"Stripping the leading spaces leaves '{stripped}'.\n"
        f"Re-wrapping gives {ans}."
    )
    return reasoning, ans


def _rand8(rng):
    return [rng.randint(0, 1) for _ in range(8)]


def aug_match(rng):
    """Synthesize a bit-column matching puzzle: identify the 2-input Boolean op."""
    name, fn = rng.choice(_BIT_OPS)
    demos = []
    for _ in range(rng.randint(3, 4)):
        a, b = _rand8(rng), _rand8(rng)
        demos.append((a, b, [fn(x, y) for x, y in zip(a, b)]))
    qa, qb = _rand8(rng), _rand8(rng)
    qo = [fn(x, y) for x, y in zip(qa, qb)]
    fmt = lambda b: "".join(map(str, b))
    demo_str = "\n".join(f"{fmt(a)}, {fmt(b)} -> {fmt(o)}" for a, b, o in demos)
    return _MATCH_TMPL.format(demos=demo_str, q=f"{fmt(qa)}, {fmt(qb)}"), fmt(qo)


def solve_match(prompt):
    """Find the single Boolean op consistent with all aligned bits, then apply it."""
    lines = re.findall(r"([01]{8}),\s*([01]{8})\s*->\s*([01]{8})", prompt)
    qm = re.search(r"output for:\s*([01]{8}),\s*([01]{8})", prompt)
    if not lines or not qm:
        return None
    A = [[int(c) for c in a] for a, b, o in lines]
    B = [[int(c) for c in b] for a, b, o in lines]
    O = [[int(c) for c in o] for a, b, o in lines]
    found = None
    for name, fn in _BIT_OPS:
        if all(O[k][j] == fn(A[k][j], B[k][j])
               for k in range(len(lines)) for j in range(8)):
            found = (name, fn); break
    if found is None:
        return None
    name, fn = found
    qa = [int(c) for c in qm.group(1)]
    qb = [int(c) for c in qm.group(2)]
    ans = "".join(str(fn(x, y)) for x, y in zip(qa, qb))
    reasoning = (
        f"Comparing the aligned input bits to the output across the examples, each "
        f"output bit is A {name} B.\n"
        f"Applying {name} bit by bit to {qm.group(1)} and {qm.group(2)} gives {ans}."
    )
    return reasoning, ans


def generate_augmented(specs, seed=0):
    """Generate verified synthetic traces; keep only solver-recovered, round-trip ones."""
    records = []
    counts = {}
    for i, (cat, generator, solver, target) in enumerate(specs):
        rng = random.Random(seed + i * 7919)
        kept = 0; tries = 0
        while kept < target and tries < target * 8:
            tries += 1
            g = generator(rng)
            if g is None:
                continue
            prompt, true = g
            out = solver(prompt)
            if out is None:
                continue
            reasoning, ans = out
            completion = f"{reasoning}\n</think>\n\\boxed{{{ans}}}"
            if not compare_answer(ans, extract_answer(completion)):
                continue
            if compare_answer(true, ans):
                records.append(dict(id=f"aug_{cat}_{kept}", category=cat, prompt=prompt,
                                    reasoning=reasoning, answer=str(true).strip()))
                kept += 1
        counts[cat] = kept
    return records, counts


_specs = []
if AUGMENT:
    _specs += [
        ("cipher", aug_cipher, solve_cipher, AUG_CIPHER),
        ("bit_manipulation", aug_bit, solve_bit, AUG_BIT),
        ("equation", aug_equation, solve_equation, AUG_EQUATION),
    ]
if AUG_SKILLS:
    _specs += [
        ("skill_spelling", aug_spell, solve_spell, AUG_SPELL),
        ("skill_concatenation", aug_concat, solve_concat, AUG_CONCAT),
        ("skill_splitting", aug_split, solve_split, AUG_SPLIT),
        ("skill_lstrip", aug_lstrip, solve_lstrip, AUG_LSTRIP),
        ("skill_matching", aug_match, solve_match, AUG_MATCH),
    ]
if _specs:
    aug_records, aug_counts = generate_augmented(_specs, seed=0)
    print("Augmented (verified synthetic) traces:")
    for c, n in aug_counts.items():
        print(f"  {c:20s} +{n}")
    corpus_records = corpus_records + aug_records
    print(f"Corpus after augmentation: {len(corpus_records)} traces")
else:
    print("Augmentation disabled.")

# --- Token-budget rebalance -----------------------------------------------------
# gravity/unit traces are procedurally IDENTICAL (one long-division/long-mult recipe;
# only the numbers differ) yet at ~1.5k tokens each the full 3.2k of them would be
# ~45% of the corpus token budget. 600 verified examples per category is plenty to
# drill the procedure; every other category is kept in full. Seeded -> reproducible.
_SUBSAMPLE = {"gravity": 600, "unit_conversion": 600}
_rng_bal = random.Random(123)
_by_cat = {}
for _r in corpus_records:
    _by_cat.setdefault(_r["category"], []).append(_r)
corpus_records = []
for _cat, _recs in _by_cat.items():
    _n = _SUBSAMPLE.get(_cat)
    if _n is not None and len(_recs) > _n:
        _recs = _rng_bal.sample(_recs, _n)
    corpus_records.extend(_recs)
print(f"Corpus after token-budget rebalance: {len(corpus_records)} traces")
print("  " + ", ".join(f"{c}={len([r for r in corpus_records if r['category']==c])}"
                       for c in sorted(_by_cat)))



Augmented (verified synthetic) traces:
  cipher               +300
  bit_manipulation     +750
  equation             +1100
  skill_spelling       +800
  skill_concatenation  +500
  skill_splitting      +500
  skill_lstrip         +300
  skill_matching       +800
Corpus after augmentation: 12768 traces


## 4 · Build the corpus — tokenize and mask

Format every trace as chat, tokenize, and mask the prompt so the training loss is computed on the completion only.

In [ ]:
# Tokenize each record with the chat template (thinking enabled) and mask the prompt tokens
# so the SFT loss falls only on the assistant completion.

import os
import glob
import torch
from torch.utils.data import Dataset

PROMPT_SUFFIX = (
    "\nPlease put your final answer inside `\\boxed{}`. "
    "For example: `\\boxed{your answer}`"
)
IGNORE_INDEX = -100


def build_example(problem_prompt: str, reasoning: str, answer: str) -> dict:
    """Tokenize one (prompt, reasoning, answer) into input_ids + prompt-masked labels."""
    messages = [{"role": "user", "content": problem_prompt + PROMPT_SUFFIX}]
    enc = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=True,
        return_dict=True,
    )
    prompt_ids = enc["input_ids"]
    completion_text = f"{reasoning}\n</think>\n\\boxed{{{answer}}}{tokenizer.eos_token}"
    completion_ids = tokenizer.encode(completion_text, add_special_tokens=False)

    input_ids = (prompt_ids + completion_ids)[:MAX_LENGTH]
    labels = ([IGNORE_INDEX] * len(prompt_ids) + completion_ids)[:MAX_LENGTH]
    return {"input_ids": input_ids, "labels": labels}


class SFTDataset(Dataset):
    def __init__(self, records):
        self.examples = [
            build_example(r["prompt"], r["reasoning"], r["answer"]) for r in records
        ]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return self.examples[i]


def collate(batch):
    """Right-pad a batch; pad tokens are ignored in the loss."""
    maxlen = max(len(x["input_ids"]) for x in batch)
    pad_id = tokenizer.pad_token_id
    input_ids, labels, attn = [], [], []
    for x in batch:
        n = maxlen - len(x["input_ids"])
        input_ids.append(x["input_ids"] + [pad_id] * n)
        labels.append(x["labels"] + [IGNORE_INDEX] * n)
        attn.append([1] * len(x["input_ids"]) + [0] * n)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "attention_mask": torch.tensor(attn, dtype=torch.long),
    }


sft_dataset = SFTDataset(corpus_records)
_tok_counts = [len(e["input_ids"]) for e in sft_dataset.examples]
print(f"SFT corpus: {len(sft_dataset)} examples; "
      f"max_len={max(_tok_counts)}, mean_len={sum(_tok_counts) // len(_tok_counts)}")

_ex = sft_dataset.examples[0]
assert len(_ex["input_ids"]) == len(_ex["labels"]), "input_ids/labels length mismatch"
assert any(l != IGNORE_INDEX for l in _ex["labels"]), "no supervised tokens"
assert _ex["labels"].count(IGNORE_INDEX) > 0, "prompt is not masked"

_chk = corpus_records[0]
assert compare_answer(_chk["answer"], extract_answer(
    f"{_chk['reasoning']}\n</think>\n\\boxed{{{_chk['answer']}}}"
)), "completion does not parse back to the stored answer"
print("Format + masking checks passed on first record.")

## 5 · Train the LoRA adapter

Supervised fine-tuning over the verified corpus, then save the adapter.

In [ ]:
# LoRA supervised fine-tuning over the verified corpus; saves the adapter when done.

import time

from transformers import Trainer, TrainingArguments

NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
MICRO_BATCH = 4
GRAD_ACCUM = 8

training_args = TrainingArguments(
    output_dir="/kaggle/working/sft_ckpt",
    per_device_train_batch_size=MICRO_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.0,
    max_grad_norm=1.0,
    bf16=True,
    optim="paged_adamw_8bit",
    logging_steps=10,
    logging_first_step=True,
    disable_tqdm=True,       
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False,
    gradient_checkpointing=False,
    seed=0,
)


# Loaded with device_map={"": 0}, so the whole model is on cuda:0. The Trainer warns
# "already on multiple devices" for ANY hf_device_map, then skips its device move;
# clearing it lets its (no-op) .to(cuda:0) run quietly. On a single GPU is_model_parallel
# and n_gpu are unaffected, so nothing else changes.
model.hf_device_map = None

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=sft_dataset,
    data_collator=collate,
)

def _fmt_dur(s):
    h, r = divmod(int(s), 3600)
    m, sec = divmod(r, 60)
    return f"{h}h {m}m {sec}s" if h else (f"{m}m {sec}s" if m else f"{sec}s")

model.config.use_cache = False
_t0 = time.perf_counter()
train_result = trainer.train()
_train_secs = time.perf_counter() - _t0
print(train_result.metrics)
print(f"Training wall-clock: {_fmt_dur(_train_secs)} "
      f"({NUM_EPOCHS} epochs over {len(sft_dataset)} examples, "
      f"{_train_secs / len(sft_dataset) / NUM_EPOCHS * 1000:.1f} ms/example/epoch).")

print(f"Saving adapter to {OUTPUT_DIR}...")
model.save_pretrained(OUTPUT_DIR)
print("Saved:", sorted(os.path.basename(p) for p in glob.glob(os.path.join(OUTPUT_DIR, "adapter*"))))

## 6 · Quick inference test

Batched generation on the public test rows as a smoke test.

In [ ]:
# Smoke test: batched generation on the public test rows.

import os
import time

import polars as pl

TEST_CANDIDATES = [
    "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/test.csv",
    "test.csv",
]
TEST_CSV = next((p for p in TEST_CANDIDATES if os.path.exists(p)), TEST_CANDIDATES[0])
test = pl.read_csv(TEST_CSV)
print(f"Loaded {test.height} test rows from {TEST_CSV}")

GEN_BATCH = 8

model.config.use_cache = True
model.eval()
tokenizer.padding_side = "left"   # left padding is required for batched decoder generation


def generate_batch(prompts):
    """Generate completions for a list of prompts at the official eval params."""
    texts = [
        tokenizer.apply_chat_template(
            [{"role": "user", "content": p + PROMPT_SUFFIX}],
            tokenize=False, add_generation_prompt=True, enable_thinking=True,
        )
        for p in prompts
    ]
    enc = tokenizer(texts, return_tensors="pt", padding=True,
                    add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **enc, max_new_tokens=MAX_LENGTH,
            do_sample=True, temperature=0.6, top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
        )
    gen = out[:, enc["input_ids"].shape[1]:]
    return [tokenizer.decode(g, skip_special_tokens=True) for g in gen]


prompts = test["prompt"].to_list()
order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))   # batch similar lengths
outputs = [""] * len(prompts)
_t0 = time.perf_counter()
for s in range(0, len(order), GEN_BATCH):
    idxs = order[s:s + GEN_BATCH]
    for i, text in zip(idxs, generate_batch([prompts[i] for i in idxs])):
        outputs[i] = text
    _done = min(s + GEN_BATCH, len(order))
    _el = time.perf_counter() - _t0
    print(f"  {_done}/{len(order)} rows  [{_el:.0f}s elapsed, "
          f"{_el / _done:.1f}s/row]")
_gen_secs = time.perf_counter() - _t0

def _fmt_dur(s):
    h, r = divmod(int(s), 3600)
    m, sec = divmod(r, 60)
    return f"{h}h {m}m {sec}s" if h else (f"{m}m {sec}s" if m else f"{sec}s")

for i in range(len(prompts)):
    print(f"{test['id'][i]} -> \\boxed{{{extract_answer(outputs[i])}}}")
print(f"\nGenerated {len(prompts)} rows in {_fmt_dur(_gen_secs)} "
      f"({_gen_secs / max(len(prompts), 1):.1f}s/row at the official eval params).")


## 7 · Package the submission

Drop the training checkpoint and zip the adapter into `submission.zip`.

In [ ]:
# Drop the training checkpoint and zip the adapter into the submission.

import glob
import os
import shutil
import subprocess

os.chdir(OUTPUT_DIR)
shutil.rmtree('sft_ckpt', ignore_errors=True)

assert os.path.exists('adapter_config.json'), 'adapter_config.json missing'
assert glob.glob('adapter_model.safetensors'), 'adapter weights missing'

subprocess.run('zip -m submission.zip *', shell=True, check=True)
print('Wrote', os.path.join(OUTPUT_DIR, 'submission.zip'))